In [1]:
# Requires: requests, beautifulsoup4
import requests
from bs4 import BeautifulSoup
import time
import re



In [2]:
URL = "https://www.amazon.in/s?k=laptop&crid=31TPTM1NJVMEC&sprefix=laptop%2Caps%2C355&ref=nb_sb_noss_2"



In [3]:
# make header to mimic a browser visit
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"
}

In [4]:
# Create a empty list to store laptop Details

data = []
print("Data list cleared")


Data list cleared


In [5]:
for page in range(1, 2):  
    params = {"k": "laptop", "page": page}
    
    # Use verify=False to bypass SSL verification (for testing purposes)
    response = requests.get(URL, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    products = soup.find_all("div", {"data-component-type": "s-search-result"})

    
    
    for product in products:
        # Extract the product title
        title_tag = product.find("h2")
        if not title_tag:
            continue
        title_text = title_tag.get_text(strip=True)

        
        # Extract the product price
        price_tag = product.find("span", {"class": "a-price-whole"})
        price_text = price_tag.get_text(strip=True) if price_tag else "N/A"
        
        # Extract the product brand
        match = re.match(r'^\W*([A-Za-z]+)', title_text)
        brand = match.group(1).upper() if match else "UNKNOWN"

        # Extract RAM (improved version)
        ram_match = re.search(r'(\d+)\s*GB\s*(RAM|DDR\d+|LPDDR\d+)', title_text, re.IGNORECASE)
        ram = ram_match.group(1) + "GB" if ram_match else "N/A"
        
        # Extract the product rating 
        rating_tag = product.find("span", class_="a-icon-alt")
        rating = "N/A"
        if rating_tag:
            rating_match = re.search(r'(\d+\.?\d*)', rating_tag.get_text())
            rating = rating_match.group(1) if rating_match else "N/A"

        # Extract the Product SSD - Simplified
        ssd_match = re.search(r'(\d+)\s*(GB|TB)\s*(?:SSD|NVMe)', title_text, re.IGNORECASE) or re.search(r'(\d+)\s*(GB|TB)', title_text)
        ssd = (ssd_match.group(1) + ssd_match.group(2).upper()) if ssd_match else "N/A"
      

        # Extract the Product Windows Version - Improved version
        windows_version = "N/A"
        
        # Try: "Windows 11", "Windows 10", "Win11", "Win10", etc.
        windows_match = re.search(r'(?:Windows\s*|Win\s*)(\d+)', title_text, re.IGNORECASE)
        if windows_match:
            windows_version = "Windows " + windows_match.group(1)
        
        # Extract the Product Color
        color_match = re.search(r'\b(Black|Silver|Gray|Grey|White|Blue|Red|Gold|Green|Brown|Pink|Purple|Yellow|Orange|Champagne|Midnight|Space|Cosmic|Stardust|Graphite|Ash|Onyx|Platinum|Metallic)\b', title_text, re.IGNORECASE)
        color = color_match.group(1) if color_match else "N/A"

        # Extract the Processor (Intel, AMD, Apple M, etc)
        processor = "N/A"
        
        # Try Intel processors - more flexible patterns
        intel_match = re.search(r'Intel\s+(?:Core\s+)?(?:i[3579]|m\d|Pentium|Celeron|Atom|Xeon)[-\w]*', title_text, re.IGNORECASE)
        if intel_match:
            processor = intel_match.group(0).strip()
        
        # Try AMD Ryzen/Athlon processors
        elif re.search(r'AMD', title_text, re.IGNORECASE):
            amd_match = re.search(r'AMD\s+(?:Ryzen|Athlon)[\s\w]*', title_text, re.IGNORECASE)
            if amd_match:
                processor = amd_match.group(0).strip()
        
        # Try Apple M series
        elif re.search(r'Apple\s+M', title_text, re.IGNORECASE):
            apple_match = re.search(r'Apple\s+M\d+\w*', title_text, re.IGNORECASE)
            if apple_match:
                processor = apple_match.group(0).strip()
        
        # Try other processors
        else:
            other_match = re.search(r'Qualcomm\s+Snapdragon|MediaTek|ARM|Exynos', title_text, re.IGNORECASE)
            if other_match:
                processor = other_match.group(0).strip()


        # Store the extracted data in a dictionary and append to the list
        data.append({
            "Title": title_text,
            "Price": price_text,
            "Brand": brand,
            'RAM': ram,
            "Rating": rating,
            "Storage": ssd,
            "Windows": windows_version,
            "Color": color,
            "Processor": processor
            
        })

      
    print(f"Page {page} scraped")
    time.sleep(1)


Page 1 scraped


In [6]:
for product in data:
    print("Title:", product["Title"])
    print("Price:", product["Price"])
    print("Brand:", product["Brand"])
    print("RAM:", product.get("RAM"))
    print("Processor:", product.get("Processor"))
    print("Rating:", product.get("Rating"))
    print("Storage:", product.get("Storage"))
    print("Windows:", product.get("Windows"))
    print("Color:", product.get("Color"))
    print("-" * 50)


Title: HP Victus, AMD Ryzen 7 7445HS, 6GB RTX 4050, 16GB DDR5(Upgradeable) 512GB SSD, 144Hz, IPS, 300nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Blue, 2.29kg, fb3130AX, Backlit, DTS Audio, Gaming Laptop
Price: 85,990
Brand: HP
RAM: 16GB
Processor: AMD Ryzen 7 7445HS
Rating: 3.9
Storage: 512GB
Windows: Windows 11
Color: Blue
--------------------------------------------------
Title: HP Smartchoice Victus, 13th Gen i7-13620H, 6GB RTX 4050, 16GB DDR4(Upgradeable) 512GB SSD, 144Hz, 300nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Mica Silver, 2.3kg, fa2100/03/04tx, Gaming Laptop
Price: 92,990
Brand: HP
RAM: 16GB
Processor: N/A
Rating: 4.0
Storage: 512GB
Windows: Windows 11
Color: Silver
--------------------------------------------------
Title: EBook 11.6" HD Laptop | Best Student & Office Work Laptop | Celeron N4020 | 4GB DDR4 | 128GB eMMC + M.2 SSD Expandable Slot | Win 11 Home |31Wh Battery | UHD Graphics 600 | Black
Price: 10,990
Brand: EBOOK
RAM: 4GB
Processor: N/A
Rating: 3.8
S

In [7]:
# make data into a dataframe 
import pandas as pd

# Create a DataFrame from the data list
df = pd.DataFrame(data)
df.head()

,Title,Price,Brand,RAM,Rating,Storage,Windows,Color,Processor
0,"HP Victus, AMD Ryzen 7 7445HS, 6GB RTX 4050, 1...","85,990",HP,16GB,3.9,512GB,Windows 11,Blue,AMD Ryzen 7 7445HS
1,"HP Smartchoice Victus, 13th Gen i7-13620H, 6GB...","92,990",HP,16GB,4.0,512GB,Windows 11,Silver,N/A
2,"EBook 11.6"" HD Laptop | Best Student & Office ...","10,990",EBOOK,4GB,3.8,4GB,Windows 11,Black,N/A
3,JioBook 11 with Lifetime Office | Android 4G L...,"10,990",JIOBOOK,4GB,2.9,4GB,N/A,Blue,Mediatek
4,"HP 15, AMD Ryzen 3 7320U (8GB DDR4, 512GB SSD)...","36,990",HP,8GB,4.0,512GB,Windows 11,Silver,AMD Ryzen 3 7320U


In [8]:
print("First 3 product titles:")
for i in range(min(3, len(data))):
    print(f"\n{i+1}. Title: {data[i]['Title']}")
    print(f"   Processor: {data[i]['Processor']}")


First 3 product titles:

1. Title: HP Victus, AMD Ryzen 7 7445HS, 6GB RTX 4050, 16GB DDR5(Upgradeable) 512GB SSD, 144Hz, IPS, 300nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Blue, 2.29kg, fb3130AX, Backlit, DTS Audio, Gaming Laptop
   Processor: AMD Ryzen 7 7445HS

2. Title: HP Smartchoice Victus, 13th Gen i7-13620H, 6GB RTX 4050, 16GB DDR4(Upgradeable) 512GB SSD, 144Hz, 300nits, FHD, 15.6''/39.6cm, Win11, M365* Office24, Mica Silver, 2.3kg, fa2100/03/04tx, Gaming Laptop
   Processor: N/A

3. Title: EBook 11.6" HD Laptop | Best Student & Office Work Laptop | Celeron N4020 | 4GB DDR4 | 128GB eMMC + M.2 SSD Expandable Slot | Win 11 Home |31Wh Battery | UHD Graphics 600 | Black
   Processor: N/A
